# OData and API Connectivity Test
用于单独验证本机环境下 Chicago 和 Cook County 数据端点是否可读。
本测试会依次尝试 resource JSON、OData v4、CSV download，并给出可用建议。

In [1]:
import os
import json
import time
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd

try:
    import requests
except Exception:
    requests = None

def fetch_json(url, params=None, timeout=40, retries=3, sleep_seconds=1.0):
    params = params or {}
    headers = {"User-Agent": "Mozilla/5.0", "Accept": "application/json"}
    app_token = os.getenv("SOCRATA_APP_TOKEN")
    if app_token:
        headers["X-App-Token"] = app_token

    last_err = None
    for i in range(retries):
        try:
            if requests is not None:
                resp = requests.get(url, params=params, headers=headers, timeout=timeout)
                resp.raise_for_status()
                return resp.json()
            sep = "&" if "?" in url else "?"
            full_url = url + (sep + urlencode(params) if params else "")
            with urlopen(Request(full_url, headers=headers), timeout=timeout) as resp:
                return json.loads(resp.read().decode("utf-8"))
        except Exception as e:
            last_err = e
            if i < retries - 1:
                time.sleep(sleep_seconds * (i + 1))
            else:
                raise last_err

def rows_from_json_payload(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict) and "value" in payload and isinstance(payload["value"], list):
        return payload["value"]
    return []

def test_json_source(dataset, source, url, params):
    try:
        payload = fetch_json(url, params=params)
        rows = rows_from_json_payload(payload)
        df = pd.DataFrame(rows)
        return {
            "dataset": dataset,
            "source": source,
            "ok": True,
            "rows": len(df),
            "columns": len(df.columns),
            "error": "",
            "preview": df.head(5),
        }
    except Exception as e:
        return {
            "dataset": dataset,
            "source": source,
            "ok": False,
            "rows": 0,
            "columns": 0,
            "error": str(e),
            "preview": pd.DataFrame(),
        }

def test_csv_source(dataset, source, url):
    try:
        df = pd.read_csv(url, nrows=10, low_memory=False)
        return {
            "dataset": dataset,
            "source": source,
            "ok": True,
            "rows": len(df),
            "columns": len(df.columns),
            "error": "",
            "preview": df.head(5),
        }
    except Exception as e:
        return {
            "dataset": dataset,
            "source": source,
            "ok": False,
            "rows": 0,
            "columns": 0,
            "error": str(e),
            "preview": pd.DataFrame(),
        }

In [2]:
tests = [
    # Chicago business licenses
    ("business_license", "resource_json", "https://data.cityofchicago.org/resource/r5kz-chrr.json", {"$limit": "20"}, "json"),
    ("business_license", "odata_v4", "https://data.cityofchicago.org/api/odata/v4/r5kz-chrr", {"$top": "20"}, "json"),
    ("business_license", "csv_download", "https://data.cityofchicago.org/api/views/r5kz-chrr/rows.csv?accessType=DOWNLOAD", None, "csv"),

    # Cook County assessed values
    ("assessed_values", "resource_json", "https://datacatalog.cookcountyil.gov/resource/uzyt-m557.json", {"$limit": "20"}, "json"),
    ("assessed_values", "odata_v4", "https://datacatalog.cookcountyil.gov/api/odata/v4/uzyt-m557", {"$top": "20"}, "json"),
    ("assessed_values", "csv_download", "https://datacatalog.cookcountyil.gov/api/views/uzyt-m557/rows.csv?accessType=DOWNLOAD", None, "csv"),
]

results = []
for dataset, source, url, params, kind in tests:
    if kind == "json":
        out = test_json_source(dataset, source, url, params)
    else:
        out = test_csv_source(dataset, source, url)
    results.append(out)
    status = "OK" if out["ok"] else "FAIL"
    print(f"[{status}] {dataset} | {source}")
    if out["ok"]:
        print(f"rows={out['rows']}, cols={out['columns']}")
    else:
        print(f"error={out['error']}")
    print("-" * 80)

summary = pd.DataFrame([{k: v for k, v in r.items() if k != "preview"} for r in results])
print("\n=== Connectivity Summary ===")
display(summary)

print("\n=== Recommended source by dataset (first successful) ===")
for dataset in summary["dataset"].unique():
    ok_rows = summary[(summary["dataset"] == dataset) & (summary["ok"] == True)]
    if len(ok_rows) > 0:
        best = ok_rows.iloc[0]
        print(f"{dataset}: use {best['source']}")
    else:
        print(f"{dataset}: no source succeeded in current network environment")

print("\n=== Preview of successful sources (first 5 rows each) ===")
for r in results:
    if r["ok"] and len(r["preview"]) > 0:
        print(f"\n{r['dataset']} | {r['source']}")
        display(r["preview"])

[OK] business_license | resource_json
rows=20, cols=41
--------------------------------------------------------------------------------
[OK] business_license | odata_v4
rows=20, cols=42
--------------------------------------------------------------------------------
[OK] business_license | csv_download
rows=10, cols=37
--------------------------------------------------------------------------------
[OK] assessed_values | resource_json
rows=20, cols=19
--------------------------------------------------------------------------------
[FAIL] assessed_values | odata_v4
error=<urlopen error [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1000)>
--------------------------------------------------------------------------------
[FAIL] assessed_values | csv_download
error=<urlopen error [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1000)>
--------------------------------------------------------------------------------

=== Conn

,dataset,source,ok,rows,columns,error
0,business_license,resource_json,True,20,41,
1,business_license,odata_v4,True,20,42,
2,business_license,csv_download,True,10,37,
3,assessed_values,resource_json,True,20,19,
4,assessed_values,odata_v4,False,0,0,<urlopen error [SSL: UNEXPECTED_EOF_WHILE_READ...
5,assessed_values,csv_download,False,0,0,<urlopen error [SSL: UNEXPECTED_EOF_WHILE_READ...



=== Recommended source by dataset (first successful) ===
business_license: use resource_json
assessed_values: use resource_json

=== Preview of successful sources (first 5 rows each) ===

business_license | resource_json


,id,license_id,account_number,site_number,legal_name,doing_business_as_name,address,city,state,zip_code,...,license_status,latitude,longitude,location,:@computed_region_vrxf_vc4k,:@computed_region_awaf_s7ux,:@computed_region_6mkv_f3dw,:@computed_region_bdys_3d7i,:@computed_region_43wa_7qmu,ssa
0,3082386-20260428,3082386,408593,18,"EL AZTECA SANCHEZ, INC.",EL AZTECA MFP VIN # 1GBHP32M5G3300758 PLATE #...,5011 W FULLERTON AVE,CHICAGO,IL,60639,...,AAI,41.9240591057,-87.7517280044,"{'latitude': '41.924059105725746', 'longitude'...",19,7,22615,306,44,NaN
1,10972-20240916,2978648,26603,1,"SASSAFRAS ENTERPRISES, INC.",SASSAFRAS ENTERPRISES,1622 W CARROLL AVE 1ST,CHICAGO,IL,60612,...,AAI,41.8879277942,-87.668009391,"{'latitude': '41.88792779418475', 'longitude':...",29,41,14917,579,46,NaN
2,3082582-20260428,3082582,514760,3,U.S. MOBILE CATERERS LLC,MCCAIN SURECRISP FOOD TRUCK VIN #: 1FCLE49L16H...,2300 S THROOP ST,CHICAGO,IL,60608,...,AAI,41.8504510243,-87.6587978557,"{'latitude': '41.85045102427', 'longitude': '-...",33,8,14920,126,26,NaN
3,2216687-20250116,2999270,34007,1,AUBURN PLASTICS LLC,AUBURN PLASTICS LLC,4535 W FULLERTON AVE 1ST,CHICAGO,IL,60639,...,AAI,41.9242092625,-87.7403783554,"{'latitude': '41.924209262492624', 'longitude'...",21,7,22615,463,17,NaN
4,3069628-20260428,3069628,16384,16,BON APPETIT MANAGEMENT CO.,BON APPETIT AT OBAMA PRESIDENTIAL CENTER,6011 S STONY ISLAND AVE PLAZA LEVEL,CHICAGO,IL,60637,...,AAI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



business_license | odata_v4


,__id,id,license_id,account_number,site_number,legal_name,doing_business_as_name,address,city,state,...,license_status,license_status_change_date,ssa,latitude,location_address,location_city,longitude,location_state,location,location_zip
0,row-nukh~mk44_4x6v,2791479-20230516,2899773,458450,2,GROVER STRATEGIES LLC,Grover Strategies,20 N CLARK ST 33 3300,CHICAGO,IL,...,AAI,None,NaN,41.882434,,,-87.631037,,"{'type': 'Point', 'coordinates': [-87.63103701...",
1,row-gh44_pjn2.yqcj,2757861-20210216,2765962,413520,8,BIG CITY OPTICAL LLC,Big City Optical,1647 N DAMEN AVE,CHICAGO,IL,...,AAI,None,33,41.911911,,,-87.677350,,"{'type': 'Point', 'coordinates': [-87.67735045...",
2,row-2pz7.3uec~ixup,2163831-20120921,2163831,373699,1,"SUNSTONE EAST GRAND LESSEE, INC.",HILTON GARDEN INN CHICAGO DOWNTOWN/MAGNIFICENT...,10 E GRAND AVE 1-23,CHICAGO,IL,...,AAI,None,NaN,41.891791,,,-87.627658,,"{'type': 'Point', 'coordinates': [-87.62765774...",
3,row-298r_j9xg-umkf,13765-20020216,1204807,30412,9,K MART CORP,BIG KMART #9355,1740 N KOSTNER AVE,CHICAGO,IL,...,AAI,None,NaN,41.912770,,,-87.736348,,"{'type': 'Point', 'coordinates': [-87.73634835...",
4,row-di93.aaaq-w9fx,8584-20220616,2842014,25897,1,PEACE PHARMACY INC,PEACE PHARMACY INC,2320 S WENTWORTH AVE 1ST,CHICAGO,IL,...,AAI,None,NaN,41.850301,,,-87.632106,,"{'type': 'Point', 'coordinates': [-87.63210603...",



business_license | csv_download


,ID,LICENSE ID,ACCOUNT NUMBER,SITE NUMBER,LEGAL NAME,DOING BUSINESS AS NAME,ADDRESS,CITY,STATE,ZIP CODE,...,LICENSE TERM START DATE,LICENSE TERM EXPIRATION DATE,LICENSE APPROVED FOR ISSUANCE,DATE ISSUED,LICENSE STATUS,LICENSE STATUS CHANGE DATE,SSA,LATITUDE,LONGITUDE,LOCATION
0,2646110-20190117,2646110,455805,1,LIVIU GEANA JR.,L & G PLUMBING,7915 N KEDVALE AVE,SKOKIE,IL,60076,...,01/17/2019,02/15/2021,01/17/2019,01/17/2019,AAI,NaN,NaN,NaN,NaN,NaN
1,1842811-20250816,3033364,320396,1,UNITED ENTERPRISE LLC,UNITED ENTERPRISE LLC,7414 W 56TH ST,SUMMIT,IL,60501,...,08/16/2025,08/15/2027,06/20/2025,06/23/2025,AAI,NaN,NaN,NaN,NaN,NaN
2,2718823-20210816,2718823,383001,4,FMS INC.,OKLAHOMA FMS INC.,1551 N 11TH AVE 150,NAMPA,ID,83687,...,08/16/2021,07/15/2021,03/16/2020,08/16/2021,AAI,NaN,NaN,NaN,NaN,NaN
3,2208295-20150516,2388936,343280,2,WALLY'S BOILER REPAIR INC.,WALLY'S BOILER REPAIR INC.,5638 W 109TH ST,CHICAGO RIDGE,IL,60415,...,05/16/2015,05/15/2017,04/07/2015,04/08/2015,AAI,NaN,NaN,NaN,NaN,NaN
4,2333607-20140527,2333607,390633,1,KEVIN MICHAEL LIPA,KEVIN MICHAEL LIPA,33441 PERIWINKLE DR,DANA POINT,CA,92629,...,05/27/2014,06/15/2016,NaN,05/27/2014,AAI,NaN,NaN,NaN,NaN,NaN



assessed_values | resource_json


,pin,year,class,township_code,township_name,nbhd,mailed_bldg,mailed_land,mailed_tot,mailed_hie,certified_bldg,certified_land,certified_tot,certified_hie,board_bldg,board_land,board_tot,board_hie,row_id
0,32213210090000,2020,202,12,Bloom,12111,2287,1007,3294,0,2287,1007,3294,0,2287,1007,3294,0,322132100900002020
1,32214030050000,2020,EX,12,Bloom,12111,0,0,0,0,0,0,0,0,0,0,0,0,322140300500002020
2,32214040370000,2020,EX,12,Bloom,12111,0,0,0,0,0,0,0,0,0,0,0,0,322140403700002020
3,32214110080000,2020,EX,12,Bloom,12111,0,0,0,0,0,0,0,0,0,0,0,0,322141100800002020
4,32214180030000,2020,EX,12,Bloom,12111,0,0,0,0,0,0,0,0,0,0,0,0,322141800300002020
